In [1]:
import os

import matplotlib.pyplot as plt

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
from typing import List, Callable, Iterable, Tuple

from gridops_multidim import BSplineInterpolationAxis, BSplineInterpolationGrid
from gridops_multidim import set_up_grid_axis

from gridops_multidim import multi_inds_from_individual_axes_inds, \
    arbitrary_dim_outer
from gridops_multidim import create_anterpolation_operator
from gridops_multidim import create_restriction_operator_1d, \
    create_restriction_operator
from gridops_multidim import create_prolongation_operator_1d, create_prolongation_operator
from gridops_multidim import create_interaction_operator
from gridops_multidim import create_compute_U_oneplus, \
    create_compute_U_and_f_oneplus

import sys

sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmfornn.splines.nesting import compute_J_zeroplus


In [2]:
%matplotlib notebook

# Basic settings

In [3]:
# geometry
length = 10.0
ndim = 3

# MSM
max_gridlevel = 4

# splines
p = 6
order = p - 1

# particles
n_particles = 5

In [4]:
J_zeroplus = compute_J_zeroplus(p)
J = jnp.concatenate((J_zeroplus[::-1][:-1], J_zeroplus))

# Create particle configuration

In [5]:
# TODO: change back to more particles and random charges
#  (few particles and hard-coded charges serve visualization purposes only)

rng = onp.random.default_rng(1632794)
pos = rng.uniform(0., length, size=(n_particles, ndim))
# chg = rng.uniform(-1., 1., size=n_particles)
chg = onp.array([-2, -2, 1, 1, 1])

# Construct grids

In [6]:
grid_axes_all_levels = [None]  # there is no grid at level zero
for lvl in range(1, max_gridlevel + 1):
    h = length / (2 ** (max_gridlevel - lvl))
    print(lvl, h)
    grid_axis = set_up_grid_axis(length=length, h=h, p=p, J_zeroplus=J_zeroplus, periodic=False)
    grid_axes_all_levels.append(grid_axis)
    
# For now, we are just replicating the same axis along all dimensions
# (i.e., same grid spacing, box size, and boundary conditions along all dimensions)
grids_all_levels = [(None, ) * ndim] + [BSplineInterpolationGrid((ga,) * ndim) for ga in grid_axes_all_levels[1:]]
grid_level_one = grids_all_levels[1]
grid_level_two = grids_all_levels[2]

1 1.25
2 2.5
3 5.0
4 10.0


# Construct kernel stencils

In [7]:
sys.path.append("/home/florian/PhD/work/code/msm_for_nn/")

from msmjax.kernels import SofteningFunctionOneOverR, split_one_over_r_kernel
from msmfornn.gridtools import construct_grids_all_levels
from msmfornn.splines.coefficients import compute_coeffs_withtruncation
from msmfornn.grid_to_grid_mapping import \
    compute_kernel_stencils_all_gridlevels

In [8]:
# We're using equal grid spacings in all directions here
level_one_gridspacing = grids_all_levels[1].axes[0].h
alpha = 2.5
level_zero_cutoff = alpha * level_one_gridspacing

params_oldmsm = {
    "min_pos": onp.array([0.] * grid_level_one.ndim),
    "max_pos": onp.array([a.length for a in grid_level_one.axes]),
    "level_one_gridspacing": level_one_gridspacing,
    "level_zero_cutoff": level_zero_cutoff,
    "max_gridlevel": max_gridlevel,
    "p": p,
    "mu": 3
}

grids_oldmsm = construct_grids_all_levels(
    min_pos=params_oldmsm["min_pos"],
    max_pos=params_oldmsm["max_pos"],
    p=params_oldmsm["p"],
    level_one_gridspacing=params_oldmsm["level_one_gridspacing"],
    max_gridlevel=params_oldmsm["max_gridlevel"],
)
partial_kernels = split_one_over_r_kernel(
    max_level=max_gridlevel,
    level_zero_cutoff=level_zero_cutoff,
    softening_function=SofteningFunctionOneOverR(p),
)
omega, _ = compute_coeffs_withtruncation(p=params_oldmsm["p"],
                                         mu=params_oldmsm["mu"])
omega_zeroplus = omega[len(omega) // 2:]

kernelstencils_old = compute_kernel_stencils_all_gridlevels(
    kernelfunctions=partial_kernels,
    grids=grids_oldmsm,
    level_zero_cutoff=level_zero_cutoff,
    omega_zeroplus=omega_zeroplus,
)
kernelstencils_old_symmetric = [None]
for stncl in kernelstencils_old[1:]:
    pw = [(s - 1, 0) for s in stncl.shape]
    stncl_symm = jnp.pad(stncl, pad_width=pw, mode='reflect')
    kernelstencils_old_symmetric.append(stncl_symm)

# Functions

## Anterpolation

In [9]:
grid_level_one.shape

(15, 15, 15)

In [10]:
@jax.jit
def evaluate_bspline_basis_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_multiparticle(pos)


@jax.jit
def evaluate_bspline_basis_gradient_multiparticle(pos):
    return grid_level_one.evaluate_bspline_basis_gradient_multiparticle(pos)

In [11]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_multiparticle(pos)

1.1 ms ± 145 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
jax.device_put(pos)
%timeit evaluate_bspline_basis_gradient_multiparticle(pos)

1.87 ms ± 68.2 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
vals, inds = evaluate_bspline_basis_multiparticle(pos)
grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)

In [14]:
@jax.jit
def combined(pos):
    vals, inds = evaluate_bspline_basis_multiparticle(pos)
    grads, _ = evaluate_bspline_basis_gradient_multiparticle(pos)
    return inds, vals, grads

In [15]:
jax.device_put(pos)
%timeit combined(pos)

2.05 ms ± 123 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [16]:
anterpolate_level_one = create_anterpolation_operator(grid=grid_level_one)

In [17]:
jitted_anterpolate = jax.jit(anterpolate_level_one)

In [18]:
jax.device_put(pos)
jax.device_put(chg)

%timeit jitted_anterpolate(pos, chg).block_until_ready()

The slowest run took 5.30 times longer than the fastest. This could mean that an intermediate result is being cached.
2.46 ms ± 1.55 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
gridcharge_level_one = anterpolate_level_one(pos, chg)

In [20]:
gridcharge_level_one

Array([0., 0., 0., ..., 0., 0., 0.], dtype=float64)

In [21]:
flat_indices = jnp.arange(gridcharge_level_one.shape[0])
unraveled_indices = jnp.unravel_index(flat_indices, grid_level_one.shape)
grid_points = []
for i, axis in enumerate(grid_level_one.axes):
    points = axis.to_raw_indices(unraveled_indices[i]) * axis.h
    grid_points.append(points)

In [22]:
mask = onp.abs(gridcharge_level_one) >= 0.01

x, y, z, = grid_points

fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(x[mask], y[mask], z[mask], c=gridcharge_level_one[mask], s=10)
ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c=chg, s=100)

plt.show()

<IPython.core.display.Javascript object>

## Restriction

In [23]:
anterpolation_funcs_all_levels = [None] + [
    jax.jit(create_anterpolation_operator(g)) for g in grids_all_levels[1:]]

restriction_funcs_all_levels = [None, None]
for lvl in range(2, len(grids_all_levels)):
    rf = create_restriction_operator(grid_source_fine=grids_all_levels[lvl - 1],
                                     grid_target_coarse=grids_all_levels[lvl])
    rf = jax.jit(rf)
    restriction_funcs_all_levels.append(rf)

In [24]:
gridcharges_direct = [None] + [af(pos, chg) for af in
                               anterpolation_funcs_all_levels[1:]]

# TODO: make output shapes of calculation via interpolation and restriction compatible
#  (the former currently returns a flat array, the latter a multi-dimensional one)
gridcharges_via_restriction = [None, gridcharges_direct[1].reshape(grids_all_levels[1].shape)]
for lvl in range(2, len(grids_all_levels)):
    rf = restriction_funcs_all_levels[lvl]
    gc_lowergrid = gridcharges_via_restriction[lvl - 1]
    gc = rf(gc_lowergrid)
    gridcharges_via_restriction.append(gc)

In [25]:
for gc_direct, gc_restrict in zip(gridcharges_direct[2:],
                                  gridcharges_via_restriction[2:]):
    assert jnp.allclose(gc_direct, gc_restrict.ravel()) # TODO: shapes

## Prolongation

In [26]:
prolongate = create_prolongation_operator(
    grid_target_fine=grids_all_levels[1],
    grid_source_coarse=grids_all_levels[2],
)

In [27]:
gridcharges_via_restriction[2].shape

(11, 11, 11)

In [28]:
prolongate(gridcharges_via_restriction[2]).shape

(15, 15, 15)

## Interaction

In [29]:
arr = arbitrary_dim_outer(jnp.arange(4), jnp.arange(1, 5))

In [30]:
arr

Array([[ 0,  0,  0,  0],
       [ 1,  2,  3,  4],
       [ 2,  4,  6,  8],
       [ 3,  6,  9, 12]], dtype=int64)

In [31]:
arr[jnp.array([0, 0])]

Array([[0, 0, 0, 0],
       [0, 0, 0, 0]], dtype=int64)

In [32]:
index_array = jnp.array([(0, 0), (1, 1), (1, 2), (2, 3)])

In [33]:
index_array

Array([[0, 0],
       [1, 1],
       [1, 2],
       [2, 3]], dtype=int64)

In [34]:
arr[index_array]

Array([[[ 0,  0,  0,  0],
        [ 0,  0,  0,  0]],

       [[ 1,  2,  3,  4],
        [ 1,  2,  3,  4]],

       [[ 1,  2,  3,  4],
        [ 2,  4,  6,  8]],

       [[ 2,  4,  6,  8],
        [ 3,  6,  9, 12]]], dtype=int64)

In [35]:
jnp.take(arr, jnp.ravel_multi_index(index_array.T, arr.shape)).T

Array([0, 2, 3, 8], dtype=int64)

In [36]:
jnp.ravel_multi_index(index_array.T, arr.shape)

Array([ 0,  5,  6, 11], dtype=int64)

In [37]:
arr

Array([[ 0,  0,  0,  0],
       [ 1,  2,  3,  4],
       [ 2,  4,  6,  8],
       [ 3,  6,  9, 12]], dtype=int64)

In [38]:
index_array

Array([[0, 0],
       [1, 1],
       [1, 2],
       [2, 3]], dtype=int64)

In [39]:
arr.at[jnp.ravel_multi_index(index_array.T, arr.shape)].get()

Array([[ 0,  0,  0,  0],
       [ 3,  6,  9, 12],
       [ 3,  6,  9, 12],
       [ 3,  6,  9, 12]], dtype=int64)

In [40]:
arr.take(jnp.ravel_multi_index(index_array.T, arr.shape))

Array([0, 2, 3, 8], dtype=int64)

In [78]:
def make_ravel_multi_inds_and_apply_bcs(grid: BSplineInterpolationGrid):
    # TODO: should this be a method of BSplineInterpolationGrid?
    is_not_periodic = ~jnp.array([ga.periodic for ga in grid.axes])
    intentionally_out_of_bounds_index = grid.size

    def ravel_multi_inds_and_apply_bcs(multi_indices: jax.Array) -> jax.Array:
        # This handles periodic axes on its own by using the "wrap" keyword
        # TODO: Does the vmapping without transposing argument give the same
        #  as argument transpose with no vmap?
        # flat_inds = jax.vmap(
        #     lambda multi_index: jnp.ravel_multi_index(
        #         multi_index, dims=grid.shape, mode="wrap"
        #     )
        # )(multi_indices)
        flat_inds = jnp.ravel_multi_index(
            multi_indices.T, dims=grid.shape, mode="wrap"
        )
        # Explicitly handle non-periodic axes
        is_out_of_bounds = jnp.logical_or(
            multi_indices < 0, multi_indices >= jnp.array(grid.shape)
        )
        is_out_of_bounds = (is_out_of_bounds & is_not_periodic).any(axis=1)
        flat_inds = jnp.where(
            is_out_of_bounds.ravel(),
            intentionally_out_of_bounds_index,
            flat_inds,
        )

        return flat_inds

    return ravel_multi_inds_and_apply_bcs


def create_interaction_operator_multidim(
    grid: BSplineInterpolationGrid, kernel_stencil: npt.ArrayLike
):
    kernel_stencil = jnp.asarray(kernel_stencil)
    kernelranges_individual_axes = [
        jnp.arange(-(s // 2), (s // 2) + 1) for s in kernel_stencil.shape
    ]

    def get_neighbor_flat_inds(multi_index: Iterable):
        neighbor_inds_individual_axes = [
            idx + kernelrange
            for idx, kernelrange in zip(
                multi_index, kernelranges_individual_axes
            )
        ]
        neighbor_multi_inds = multi_inds_from_individual_axes_inds(
            *neighbor_inds_individual_axes
        )
        return ravel_multi_inds_and_apply_bcs(neighbor_multi_inds)
    
    multi_inds_all_points = multi_inds_from_individual_axes_inds(*[jnp.arange(s) for s in grid.shape])
    
    def apply_interaction(in_array):
        pass
    
    return apply_interaction
    

In [42]:
kernelstencil_level_one = kernelstencils_old_symmetric[1] 

# kernelranges_individual_axes = [s // 2 for s in kernelstencil_level_one.shape]
kernelranges_individual_axes = [jnp.arange(-(s // 2), (s // 2) + 1) for s in kernelstencil_level_one.shape]


In [43]:
kernelranges_individual_axes

[Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64),
 Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64),
 Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64)]

In [44]:
grid_level_one.shape

(15, 15, 15)

In [45]:
idx = 8
idx + kernelranges_individual_axes[0]

Array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14], dtype=int64)

In [46]:
multi_index = (0, 0, 0)

neighbor_indices_individual_axes = [idx + kernelrange for idx, kernelrange in zip(multi_index, kernelranges_individual_axes)]
neighbor_indices_individual_axes

[Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64),
 Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64),
 Array([-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6], dtype=int64)]

In [47]:
neighbor_multi_indices = multi_inds_from_individual_axes_inds(*neighbor_indices_individual_axes)

In [80]:
ravel_multi_inds_and_apply_bcs = make_ravel_multi_inds_and_apply_bcs(grid_level_one)

In [49]:
neighbor_flat_indices = ravel_multi_inds_and_apply_bcs(neighbor_multi_indices)

In [50]:
neighbor_flat_indices.shape

(2197,)

In [51]:
(neighbor_flat_indices == grid_level_one.size).sum()

Array(1854, dtype=int64)

In [52]:
neighbor_flat_indices.size * (7 / 8)

1922.375

In [53]:
jnp.ravel_multi_index(neighbor_indices_individual_axes, dims=grid_level_one.shape, mode="wrap")

Array([2169, 2410, 2651, 2892, 3133, 3374,    0,  241,  482,  723,  964,
       1205, 1446], dtype=int64)

In [54]:
jnp.array(neighbor_indices_individual_axes)

Array([[-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6],
       [-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6],
       [-6, -5, -4, -3, -2, -1,  0,  1,  2,  3,  4,  5,  6]], dtype=int64)

In [56]:
(get_neighbor_flat_inds((0, 0, 0)) == grid_level_one.size).sum()

Array(1854, dtype=int64)

In [57]:
grid_level_one.size - jnp.prod(jnp.array([(s // 2) + 1 for s in kernelstencil_level_one.shape]))

Array(3032, dtype=int64)

In [58]:
multi_inds_all_points = multi_inds_from_individual_axes_inds(*[jnp.arange(s) for s in grid_level_one.shape])

In [59]:
multi_inds_all_points.shape

(3375, 3)

In [81]:
ravel_multi_inds_and_apply_bcs(multi_inds_all_points[0])

ValueError: axis 1 is out of bounds for array of dimension 1

In [82]:
jax.vmap(ravel_multi_inds_and_apply_bcs)(multi_inds_all_points).shape

ValueError: axis 1 is out of bounds for array of dimension 1

In [69]:
jnp.ravel_multi_index((0, 0, 0), dims=grid_level_one.shape)

Array(0, dtype=int64)

In [72]:
index_array

Array([[0, 0],
       [1, 1],
       [1, 2],
       [2, 3]], dtype=int64)

In [73]:
jnp.ravel_multi_index(jnp.array([[0, 0, 0], [1, 0, 0]]).T, dims=grid_level_one.shape)

Array([  0, 225], dtype=int64)

In [74]:
def ravel_fun(index_array):
    return jnp.ravel_multi_index(index_array.T, dims=grid_level_one.shape)

In [76]:
ravel_fun(jnp.array([1, 0, 0]))

Array(225, dtype=int64)

In [77]:
ravel_fun(jnp.array([[1, 0, 0], [0, 0, 1]]))

Array([225,   1], dtype=int64)